In [ ]:
# Comment all these 
# import os
# del os.environ['CUDA_VISIBLE_DEVICES']
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

In [385]:
%matplotlib inline

In [386]:
!pip install torchmetrics scikit-learn tqdm rouge-score numpy

Translation with a Sequence to Sequence Network
*************************************************************

::

    [KEY: > input, = target, < output]

    > il est en train de peindre un tableau .
    = he is painting a picture .
    < he is painting a picture .

    > pourquoi ne pas essayer ce vin delicieux ?
    = why not try that delicious wine ?
    < why not try that delicious wine ?

    > elle n est pas poete mais romanciere .
    = she is not a poet but a novelist .
    < she not not a poet but a novelist .

    > vous etes trop maigre .
    = you re too skinny .
    < you re all alone .

... to varying degrees of success.

This is made possible by the simple but powerful idea of the `sequence
to sequence network <http://arxiv.org/abs/1409.3215>`__, in which two
recurrent neural networks work together to transform one sequence to
another. An encoder network condenses an input sequence into a vector,
and a decoder network unfolds that vector into a new sequence.

.. figure:: /_static/img/seq-seq-images/seq2seq.png
   :alt:

To improve upon this model we'll use an `attention
mechanism <https://arxiv.org/abs/1409.0473>`__, which lets the decoder
learn to focus over a specific range of the input sequence.

**Recommended Reading:**

I assume you have at least installed PyTorch, know Python, and
understand Tensors:

-  http://pytorch.org/ For installation instructions
-  :doc:`/beginner/deep_learning_60min_blitz` to get started with PyTorch in general
-  :doc:`/beginner/pytorch_with_examples` for a wide and deep overview
-  :doc:`/beginner/former_torchies_tutorial` if you are former Lua Torch user


It would also be useful to know about Sequence to Sequence networks and
how they work:

-  `Learning Phrase Representations using RNN Encoder-Decoder for
   Statistical Machine Translation <http://arxiv.org/abs/1406.1078>`__
-  `Sequence to Sequence Learning with Neural
   Networks <http://arxiv.org/abs/1409.3215>`__
-  `Neural Machine Translation by Jointly Learning to Align and
   Translate <https://arxiv.org/abs/1409.0473>`__
-  `A Neural Conversational Model <http://arxiv.org/abs/1506.05869>`__

You will also find the previous tutorials on
:doc:`/intermediate/char_rnn_classification_tutorial`
and :doc:`/intermediate/char_rnn_generation_tutorial`
helpful as those concepts are very similar to the Encoder and Decoder
models, respectively.

And for more, read the papers that introduced these topics:

-  `Learning Phrase Representations using RNN Encoder-Decoder for
   Statistical Machine Translation <http://arxiv.org/abs/1406.1078>`__
-  `Sequence to Sequence Learning with Neural
   Networks <http://arxiv.org/abs/1409.3215>`__
-  `Neural Machine Translation by Jointly Learning to Align and
   Translate <https://arxiv.org/abs/1409.0473>`__
-  `A Neural Conversational Model <http://arxiv.org/abs/1506.05869>`__


**Requirements**



Loading data files
==================

The data for this project is a set of many thousands of English to
French translation pairs.

`This question on Open Data Stack
Exchange <http://opendata.stackexchange.com/questions/3888/dataset-of-sentences-translated-into-many-languages>`__
pointed me to the open translation site http://tatoeba.org/ which has
downloads available at http://tatoeba.org/eng/downloads - and better
yet, someone did the extra work of splitting language pairs into
individual text files here: http://www.manythings.org/anki/

The English to French pairs are too big to include in the repo, so
download to ``data/eng-fra.txt`` before continuing. The file is a tab
separated list of translation pairs:

::

    I am cold.    J'ai froid.

.. Note::
   Download the data from
   `here <https://download.pytorch.org/tutorial/data.zip>`_
   and extract it to the current directory.



In [387]:
# !wget http://www.manythings.org/anki/fra-eng.zip
# !unzip -o fra-eng.zip
# !mkdir -p data
# !mv fra.txt data/eng-fra.txt

In [388]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

# torch.backends.cudnn.enabled = False
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda")

In [389]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

print(f"Environment Variable: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"PyTorch Detected Count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

Is CUDA available? True
GPU Name: NVIDIA GeForce RTX 3060
Number of GPUs: 2
Environment Variable: 0,1
PyTorch Detected Count: 2
Device 0: NVIDIA GeForce RTX 3060
Device 1: Quadro P620


The files are all in Unicode, to simplify we will turn Unicode
characters to ASCII, make everything lowercase, and trim most
punctuation.




In [390]:
# Turn a Unicode string to plain ASCII, thanks to
# http://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters


def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

To read the data file we will split the file into lines, and then split
lines into pairs. The files are all English → Other Language, so if we
want to translate from Other Language → English I added the ``reverse``
flag to reverse the pairs.




In [391]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')[:2]] for l in lines]

    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

Since there are a *lot* of example sentences and we want to train
something quickly, we'll trim the data set to only relatively short and
simple sentences. Here the maximum length is 10 words (that includes
ending punctuation) and we're filtering to sentences that translate to
the form "I am" or "He is" etc. (accounting for apostrophes replaced
earlier).




In [424]:
MAX_LENGTH = 10

eng_prefixes = (
    "i am", "i m",
    "he is", "he s",
    "she is", "she s",
    "you are", "you re",
    "we are", "we re",
    "they are", "they re"
)


def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

The full process for preparing the data is:

-  Read text file and split into lines, split lines into pairs
-  Normalize text, filter by length and content
-  Make word lists from sentences in pairs




In [425]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
#print(random.choice(pairs))

Reading lines...
Read 240521 sentence pairs
Trimmed to 18847 sentence pairs
Counting words...
Counted words:
fra 6198
eng 4070


In [394]:
from sklearn.model_selection import train_test_split

In [395]:
X = [i[0] for i in pairs]
y = [i[1] for i in pairs]

In [396]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [397]:
train_pairs = list(zip(X_train,y_train))
test_pairs = list(zip(X_test,y_test))

# Logger Utilities

In [398]:
import sys

class Logger(object):
    def __init__(self, filename='training_log.txt'):
        self.terminal = sys.stdout
        self.log = open(filename, 'a')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        # This flush method is needed for python 3 compatibility.
        # This handles the flush command by doing nothing.
        pass

# To start logging to a file:
# sys.stdout = Logger('experiment_results.txt')

print('Logger class defined. To use it, set sys.stdout = Logger("your_file.txt")')

Logger class defined. To use it, set sys.stdout = Logger("your_file.txt")


The Seq2Seq Model
=================

A Recurrent Neural Network, or RNN, is a network that operates on a
sequence and uses its own output as input for subsequent steps.

A `Sequence to Sequence network <http://arxiv.org/abs/1409.3215>`__, or
seq2seq network, or `Encoder Decoder
network <https://arxiv.org/pdf/1406.1078v3.pdf>`__, is a model
consisting of two RNNs called the encoder and decoder. The encoder reads
an input sequence and outputs a single vector, and the decoder reads
that vector to produce an output sequence.

Unlike sequence prediction with a single RNN, where every input
corresponds to an output, the seq2seq model frees us from sequence
length and order, which makes it ideal for translation between two
languages.

Consider the sentence "Je ne suis pas le chat noir" → "I am not the
black cat". Most of the words in the input sentence have a direct
translation in the output sentence, but are in slightly different
orders, e.g. "chat noir" and "black cat". Because of the "ne/pas"
construction there is also one more word in the input sentence. It would
be difficult to produce a correct translation directly from the sequence
of input words.

With a seq2seq model the encoder creates a single vector which, in the
ideal case, encodes the "meaning" of the input sequence into a single
vector — a single point in some N dimensional space of sentences.




The Encoder
-----------

The encoder of a seq2seq network is a RNN that outputs some value for
every word from the input sentence. For every input word the encoder
outputs a vector and a hidden state, and uses the hidden state for the
next input word.






In [399]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [400]:
class EncoderRNNLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNNLSTM, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        # TASK 3: Changed GRU to LSTM
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        # LSTM returns (output, (hidden, cell))
        output, hidden = self.lstm(output, hidden)
        return output, hidden

    def initHidden(self):
        # LSTM needs a tuple of (h_0, c_0)
        return (torch.zeros(1, 1, self.hidden_size, device=device),
                torch.zeros(1, 1, self.hidden_size, device=device))

In [401]:
class EncoderRNNBiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNNBiLSTM, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        # TASK 3: Changed GRU to LSTM
        self.lstm = nn.LSTM(hidden_size, hidden_size, bidirectional = True)
        self.fc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        # LSTM returns (output, (hidden, cell))
        output, hidden = self.lstm(output, hidden)
        output = self.fc(output) 
        
        return output, hidden

    def initHidden(self):
        # LSTM needs a tuple of (h_0, c_0)
        return (torch.zeros(2, 1, self.hidden_size, device=device),
                torch.zeros(2, 1, self.hidden_size, device=device))

In [402]:
# import math

# class TransformerEncoder(nn.Module):
#     def __init__(self, input_size, hidden_size, nhead=8, num_layers=3):
#         super(TransformerEncoder, self).__init__()
#         self.hidden_size = hidden_size
#         self.embedding = nn.Embedding(input_size, hidden_size)
        
#         # Positional Encoding is required for Transformers to know word order
#         self.pos_encoder = nn.Parameter(torch.zeros(1, MAX_LENGTH, hidden_size))
        
#         encoder_layers = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=nhead, batch_first=True)
#         self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)

#     def forward(self, input_tensor):
#         # Transformer expects the whole sentence at once 
#         # input_tensor shape: [seq_len, 1] -> [1, seq_len]
#         input_tensor = input_tensor.transpose(0, 1)
        
#         embedded = self.embedding(input_tensor) * math.sqrt(self.hidden_size)
#         embedded = embedded + self.pos_encoder[:, :embedded.size(1), :]
        
#         # output shape: [1, seq_len, hidden_size]
#         output = self.transformer_encoder(embedded)
        
#         # Take the mean of all token representations 
#         # This creates a single [1, 1, hidden_size] vector for the GRU Decoder
#         sentence_rep = output.mean(dim=1, keepdim=True)
        
#         # We return sentence_rep as the 'hidden' state for the GRU
#         return output, sentence_rep.transpose(0, 1)

In [403]:
class TransformerEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, nhead=8, num_layers=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        
        # Transformer layers
        encoder_layers = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)

    def forward(self, input_tensor):
        # input_tensor is [seq_len, 1] -> convert to [1, seq_len] for batch_first
        input_tensor = input_tensor.transpose(0, 1)
        
        embedded = self.embedding(input_tensor)
        
        # output shape: [1, seq_len, hidden_size]
        output = self.transformer_encoder(embedded)
        
        # TASK 6 REQUIREMENT: Take the mean of all hidden representations 
        # sentence_rep shape: [1, 1, hidden_size]
        sentence_rep = output.mean(dim=1, keepdim=True)
        
        # The GRU Decoder expects the hidden state as [1, 1, hidden_size] [cite: 181]
        # We return (Full Sequence, Mean Pooled Vector)
        return output, sentence_rep.transpose(0, 1)

The Decoder (Your assignment)
-----------

The decoder is another RNN that takes the encoder output vector(s) and
outputs a sequence of words to create the translation.




Simple Decoder
^^^^^^^^^^^^^^

In the simplest seq2seq decoder we use only last output of the encoder.
This last output is sometimes called the *context vector* as it encodes
context from the entire sequence. This context vector is used as the
initial hidden state of the decoder.

At every step of decoding, the decoder is given an input token and
hidden state. The initial input token is the start-of-string ``<SOS>``
token, and the first hidden state is the context vector (the encoder's
last hidden state).


In [404]:
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):

        # Your code here #
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [405]:
class DecoderLSTM(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderLSTM, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        # TASK 3: Changed GRU to LSTM
        self.lstm = nn.LSTM(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        # LSTM returns (output, (hidden, cell))
        output, hidden = self.lstm(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        # LSTM needs a tuple of (h_0, c_0)
        return (torch.zeros(1, 1, self.hidden_size, device=device),
                torch.zeros(1, 1, self.hidden_size, device=device))

In [406]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        # Calculate Attention Weights
        attn_weights = F.softmax(
            self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
        
        # Apply Attention Weights to Encoder Outputs to get Context
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)

        output = F.relu(output)
        output, hidden = self.gru(output, hidden)

        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

<div class="alert alert-info"><h4>Note</h4><p>There are other forms of attention that work around the length
  limitation by using a relative position approach. Read about "local
  attention" in `Effective Approaches to Attention-based Neural Machine
  Translation <https://arxiv.org/abs/1508.04025>`__.</p></div>

Training
========

Preparing Training Data
-----------------------

To train, for each pair we will need an input tensor (indexes of the
words in the input sentence) and target tensor (indexes of the words in
the target sentence). While creating these vectors we will append the
EOS token to both sequences.




In [407]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

Training the Model
------------------

To train we run the input sentence through the encoder, and keep track
of every output and the latest hidden state. Then the decoder is given
the ``<SOS>`` token as its first input, and the last hidden state of the
encoder as its first hidden state.

"Teacher forcing" is the concept of using the real target outputs as
each next input, instead of using the decoder's guess as the next input.
Using teacher forcing causes it to converge faster but `when the trained
network is exploited, it may exhibit
instability <http://minds.jacobs-university.de/sites/default/files/uploads/papers/ESNTutorialRev.pdf>`__.

You can observe outputs of teacher-forced networks that read with
coherent grammar but wander far from the correct translation -
intuitively it has learned to represent the output grammar and can "pick
up" the meaning once the teacher tells it the first few words, but it
has not properly learned how to create the sentence from the translation
in the first place.

Because of the freedom PyTorch's autograd gives us, we can randomly
choose to use teacher forcing or not with a simple if statement. Turn
``teacher_forcing_ratio`` up to use more of it.




In [408]:
# teacher_forcing_ratio = 0.5

# def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
#     encoder_hidden = encoder.initHidden()
#     # en_h, en_c = encoder_hidden

#     encoder_optimizer.zero_grad()
#     decoder_optimizer.zero_grad()

#     input_length = input_tensor.size(0)
#     target_length = target_tensor.size(0)

#     encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

#     loss = 0

#     for ei in range(input_length):
#         encoder_output, encoder_hidden = encoder(
#             input_tensor[ei], encoder_hidden)
#         encoder_outputs[ei] = encoder_output[0, 0]

#     decoder_input = torch.tensor([[SOS_token]], device=device)

#     # For Task 1-3
#     decoder_hidden = encoder_hidden

#     # For Task 4 BiLSTM + GRU
#     # decoder_hidden = en_h[0:1] + en_h[1:2]

#     use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

#     # if use_teacher_forcing:
#     #     # Teacher forcing: Feed the target as the next input
#     #     for di in range(target_length):
#     #         decoder_output, decoder_hidden = decoder(
#     #             decoder_input, decoder_hidden)
#     #         loss += criterion(decoder_output, target_tensor[di])
#     #         decoder_input = target_tensor[di]  # Teacher forcing

#     # else:
#     #     # Without teacher forcing: use its own predictions as the next input
#     #     for di in range(target_length):
#     #         decoder_output, decoder_hidden = decoder(
#     #             decoder_input, decoder_hidden)
#     #         topv, topi = decoder_output.topk(1)
#     #         decoder_input = topi.squeeze().detach()  # detach from history as input

#     #         loss += criterion(decoder_output, target_tensor[di])
#     #         if decoder_input.item() == EOS_token:
#     #             break

#     for di in range(target_length):
#         # Pass encoder_outputs for attention
#         decoder_output, decoder_hidden, decoder_attention = decoder(
#             decoder_input, decoder_hidden, encoder_outputs)
        
#         loss += criterion(decoder_output, target_tensor[di])
        
#         if use_teacher_forcing:
#             decoder_input = target_tensor[di]
#         else:
#             topv, topi = decoder_output.topk(1)
#             decoder_input = topi.squeeze().detach()
#             if decoder_input.item() == EOS_token:
#                 break

#     loss.backward()

#     encoder_optimizer.step()
#     decoder_optimizer.step()

#     return loss.item() / target_length

In [409]:
teacher_forcing_ratio = 0.5

def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    # TASK 6: Pass the WHOLE tensor to the Transformer at once 
    encoder_outputs_full, encoder_hidden = encoder(input_tensor)

    # Prepare encoder_outputs for the decoder (padding if necessary)
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
    curr_len = encoder_outputs_full.size(1)
    encoder_outputs[:curr_len] = encoder_outputs_full[0]

    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden # The mean-pooled representation

    loss = 0
    target_length = target_tensor.size(0)
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    for di in range(target_length):
        # We use the original GRU Decoder (or AttnDecoder if you prefer)
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        loss += criterion(decoder_output, target_tensor[di])
        
        if use_teacher_forcing:
            decoder_input = target_tensor[di]
        else:
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length

This is a helper function to print time elapsed and estimated time
remaining given the current time and progress %.




In [410]:
import time
import math


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

The whole training process looks like this:

-  Start a timer
-  Initialize optimizers and criterion
-  Create set of training pairs
-  Start empty losses array for plotting

Then we call ``train`` many times and occasionally print the progress (%
of examples, time so far, estimated time) and average loss.




In [411]:
# Initiate Logger param

logtime = time.strftime("%Y-%m-%d_%H%M%S")
original_stdout = sys.stdout

taskname = "task_6"
dirname = f'logs/{taskname}-{logtime}/'
os.makedirs(os.path.dirname(dirname), exist_ok = True)

In [412]:
def trainIters(encoder, decoder, epochs, print_every=1000, plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)

    criterion = nn.NLLLoss()

    iter = 1
    n_iters = len(train_pairs) * epochs
    
    try:
        sys.stdout = Logger(dirname + "_training.log.txt")
        
        for epoch in range(epochs):
            print("Epoch: %d/%d" % (epoch, epochs))
            for training_pair in train_pairs:
                training_pair = tensorsFromPair(training_pair)
    
                input_tensor = training_pair[0]
                target_tensor = training_pair[1]
    
                loss = train(input_tensor, target_tensor, encoder,
                            decoder, encoder_optimizer, decoder_optimizer, criterion)
                print_loss_total += loss
                plot_loss_total += loss
    
                if iter % print_every == 0:
                    print_loss_avg = print_loss_total / print_every
                    print_loss_total = 0
                    print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                                iter, iter / n_iters * 100, print_loss_avg))
    
                iter +=1
    finally:
        sys.stdout = original_stdout

Evaluation
==========

Evaluation is mostly the same as training, but there are no targets so
we simply feed the decoder's predictions back to itself for each step.
Every time it predicts a word we add it to the output string, and if it
predicts the EOS token we stop there. We also store the decoder's
attention outputs for display later.




In [413]:
# def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
#     with torch.no_grad():
#         input_tensor = tensorFromSentence(input_lang, sentence)
#         input_length = input_tensor.size()[0]
#         # encoder_hidden = encoder.initHidden()
#         # en_h, en_c = encoder_hidden

#         encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

#         for ei in range(input_length):
#             encoder_output, encoder_hidden = encoder(input_tensor[ei],
#                                                      encoder_hidden)
#             encoder_outputs[ei] += encoder_output[0, 0]

#         decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

#         # decoder_hidden = en_h[0:1] + en_h[1:2]
#         decoder_hidden = encoder_hidden

#         decoded_words = []
#         decoder_attentions = torch.zeros(max_length, max_length)

#         for di in range(max_length):
#             # decoder_output, decoder_hidden = decoder(
#             #     decoder_input, decoder_hidden)
#             decoder_output, decoder_hidden, decoder_attention = decoder(
#                 decoder_input, decoder_hidden, encoder_outputs)
#             decoder_attentions[di] = decoder_attention.data
            
#             topv, topi = decoder_output.data.topk(1)
#             if topi.item() == EOS_token:
#                 decoded_words.append('<EOS>')
#                 break
#             else:
#                 decoded_words.append(output_lang.index2word[topi.item()])

#             decoder_input = topi.squeeze().detach()

#         return decoded_words

In [414]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        # input_tensor is [seq_len, 1]

        # TASK 6: No loop, no initHidden. Pass the whole sequence.
        # transformer_outputs: [1, seq_len, hidden_size]
        # pooled_hidden: [1, 1, hidden_size] (the mean representation)
        transformer_outputs, pooled_hidden = encoder(input_tensor)

        # Prepare encoder_outputs for the decoder
        # We fill a zeros tensor to maintain the MAX_LENGTH expected by Attention/logging
        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
        curr_len = min(transformer_outputs.size(1), max_length)
        encoder_outputs[:curr_len] = transformer_outputs[0, :curr_len]

        decoder_input = torch.tensor([[SOS_token]], device=device)
        
        # Initialize GRU Decoder with the mean-pooled vector from Transformer
        decoder_hidden = pooled_hidden

        decoded_words = []
        for di in range(max_length):
            # If using original Decoder:
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            
            # OR if you are using AttnDecoder:
            # decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)

            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words

We can evaluate random sentences from the training set and print out the
input, target, and output to make some subjective quality judgements:




In [415]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words = evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [416]:
from torchmetrics.text.rouge import ROUGEScore
from tqdm import tqdm
import numpy as np

def inference(encoder, decoder, testing_pairs):
    input = []
    gt = []
    predict = []

    from tqdm import tqdm
    for i in tqdm(range(len(testing_pairs))):
        pair = testing_pairs[i]
        output_words = evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)

        input.append(pair[0])
        gt.append(pair[1])
        predict.append(output_sentence)

    return input,gt,predict


def eval(gt, predict):
    try:
        sys.stdout = Logger(dirname + "_eval.log.txt")
        rouge = ROUGEScore()
        metric_score = rouge(predict, gt)
        print("=== Evaluation score - Rouge score ===")
        print("Rouge1 fmeasure:\t",metric_score["rouge1_fmeasure"].item())
        print("Rouge1 precision:\t",metric_score["rouge1_precision"].item())
        print("Rouge1 recall:  \t",metric_score["rouge1_recall"].item())
        print("Rouge2 fmeasure:\t",metric_score["rouge2_fmeasure"].item())
        print("Rouge2 precision:\t",metric_score["rouge2_precision"].item())
        print("Rouge2 recall:  \t",metric_score["rouge2_recall"].item())
        print("=====================================")
    finally:
        sys.stdout = original_stdout

Training and Evaluating
=======================

With all these helper functions in place (it looks like extra work, but
it makes it easier to run multiple experiments) we can actually
initialize a network and start training.

Remember that the input sentences were heavily filtered. For this small
dataset we can use relatively small networks of 256 hidden nodes and a
single GRU layer. After about 40 minutes on a MacBook CPU we'll get some
reasonable results.

.. Note::
   If you run this notebook you can train, interrupt the kernel,
   evaluate, and continue training later. Comment out the lines where the
   encoder and decoder are initialized and run ``trainIters`` again.




In [ ]:
hidden_size = 256
# Task 1 and 2
# encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
# decoder1 = Decoder(hidden_size, output_lang.n_words).to(device)

# Task 3
# encoder1 = EncoderRNNLSTM(input_lang.n_words, hidden_size).to(device)
# decoder1 = DecoderLSTM(hidden_size, output_lang.n_words).to(device)

# Task 4
# encoder1 = EncoderRNNBiLSTM(input_lang.n_words, hidden_size).to(device)
# decoder1 = Decoder(hidden_size, output_lang.n_words).to(device)

# Task 5
# encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
# decoder1 = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

# Task 6
encoder1 = TransformerEncoder(input_lang.n_words, hidden_size).to(device)
decoder1 = Decoder(hidden_size, output_lang.n_words).to(device)

trainIters(encoder1, decoder1, 10, print_every=2000)

Epoch: 0/10
0m 37s (- 67m 16s) (2000 0%) 3.3881


In [ ]:
evaluateRandomly(encoder1, decoder1)

In [ ]:
# input,gt,predict = inference(encoder1, decoder1, train_pairs)
# eval(gt, predict)

In [ ]:
input,gt,predict = inference(encoder1, decoder1, test_pairs)
eval(gt, predict)

In [ ]:
# output.show()